# 08 — Multimodal Retrieval Pipeline

Implements the full **multimodal candidate retrieval** layer using real FAISS indexes and real CLIP embeddings.

**Supported query modes:**
| Mode | Input | Index used |
|---|---|---|
| Text only | text query string | `text_index.faiss` |
| Image only | image file path | `image_index.faiss` |
| Text + Image | both | both indexes, merged candidate pool |

**This stage outputs: a candidate pool.**  
Score fusion / re-ranking is the next stage — not implemented here.

```
USER
  ├─ TEXT  → CLIP Text Encoder  → Text FAISS  → Text Candidates
  └─ IMAGE → CLIP Image Encoder → Image FAISS → Image Candidates
                                                       ↓
                                              Candidate Pool
                                                       ↓
                                              NEXT: Re-ranking
```

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"faiss   : {faiss.__version__}")
print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}")

faiss   : 1.15.0
torch   : 2.14.0+cpu
device  : cpu


## 2. Paths — Resolved Relative to This Notebook

In [2]:
# Notebook lives in notebooks/ — all paths relative to project root
NOTEBOOK_DIR = Path(".").resolve()          # notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent          # project root
PROCESSED    = PROJECT_ROOT / "data" / "processed"
FAISS_DIR    = PROCESSED / "faiss"

# Input files
PRODUCTS_CSV        = PROCESSED / "products_ml_ready.csv"
TEXT_FAISS_PATH     = FAISS_DIR  / "text_index.faiss"
IMAGE_FAISS_PATH    = FAISS_DIR  / "image_index.faiss"
TEXT_MAPPING_CSV    = FAISS_DIR  / "text_index_mapping.csv"
IMAGE_MAPPING_CSV   = FAISS_DIR  / "image_index_mapping.csv"

for p in [PRODUCTS_CSV, TEXT_FAISS_PATH, IMAGE_FAISS_PATH,
          TEXT_MAPPING_CSV, IMAGE_MAPPING_CSV]:
    assert p.exists(), f"Required file missing: {p}"
    print(f"  OK  {p.relative_to(PROJECT_ROOT)}")

  OK  data\processed\products_ml_ready.csv
  OK  data\processed\faiss\text_index.faiss
  OK  data\processed\faiss\image_index.faiss
  OK  data\processed\faiss\text_index_mapping.csv
  OK  data\processed\faiss\image_index_mapping.csv


## 3. Load Indexes, Mappings, and Product Catalogue

In [3]:
# ── FAISS indexes ─────────────────────────────────────────────────────────────
text_index  = faiss.read_index(str(TEXT_FAISS_PATH))
image_index = faiss.read_index(str(IMAGE_FAISS_PATH))

# ── Mappings: faiss_index (int) → pid (str) ──────────────────────────────────
text_mapping_df  = pd.read_csv(TEXT_MAPPING_CSV)
image_mapping_df = pd.read_csv(IMAGE_MAPPING_CSV)

text_faiss_to_pid  = dict(zip(text_mapping_df["faiss_index"],  text_mapping_df["pid"]))
image_faiss_to_pid = dict(zip(image_mapping_df["faiss_index"], image_mapping_df["pid"]))

# ── Product metadata — indexed by pid ────────────────────────────────────────
products_df = pd.read_csv(PRODUCTS_CSV)
products_by_pid = products_df.set_index("pid")

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert text_index.d  == 512, f"text FAISS dim mismatch: {text_index.d}"
assert image_index.d == 512, f"image FAISS dim mismatch: {image_index.d}"
assert text_index.ntotal  == len(products_df), "text ntotal mismatch"
assert image_index.ntotal == len(products_df), "image ntotal mismatch"
assert len(text_mapping_df)  == len(products_df), "text mapping length mismatch"
assert len(image_mapping_df) == len(products_df), "image mapping length mismatch"
assert products_df["pid"].duplicated().sum() == 0, "duplicate PIDs in products"

N_PRODUCTS  = len(products_df)
FAISS_DIM   = text_index.d

print(f"Products          : {N_PRODUCTS}")
print(f"FAISS dim         : {FAISS_DIM}")
print(f"text_index ntotal : {text_index.ntotal}")
print(f"image_index ntotal: {image_index.ntotal}")
print(f"Text mapping rows : {len(text_mapping_df)}")
print(f"Image mapping rows: {len(image_mapping_df)}")
print("All sanity checks passed.")

Products          : 4681
FAISS dim         : 512
text_index ntotal : 4681
image_index ntotal: 4681
Text mapping rows : 4681
Image mapping rows: 4681
All sanity checks passed.


## 4. Load CLIP Model

Same model used to generate `clip_text_embeddings.npy` and `image_embeddings.npy`.

In [4]:
MODEL_NAME = "openai/clip-vit-base-patch32"

print(f"Loading {MODEL_NAME} ...")
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()
print(f"Model ready on: {DEVICE}")

Loading openai/clip-vit-base-patch32 ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model ready on: cpu


## 5. Query Encoders

Two separate encoders that both output 512-dim L2-normalized vectors.

In [5]:
def _l2_normalize(vec: np.ndarray) -> np.ndarray:
    """L2-normalize a (1, D) or (N, D) float32 array."""
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    return vec / np.clip(norm, 1e-10, None)


def encode_text(query: str) -> np.ndarray:
    """
    Encode a text string into a 512-dim normalized CLIP embedding.

    Parameters
    ----------
    query : str — natural-language search query

    Returns
    -------
    np.ndarray shape (1, 512), dtype float32, L2-normalized
    """
    if not query or not query.strip():
        raise ValueError("Text query must be a non-empty string.")

    tokens = clip_tokenizer(
        [query.strip()],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )
    tokens = {k: v.to(DEVICE) for k, v in tokens.items()}

    with torch.no_grad():
        text_out  = clip_model.text_model(**tokens)
        embedding = clip_model.text_projection(text_out.pooler_output)  # (1, 512)

    vec = embedding.cpu().float().numpy()
    return _l2_normalize(vec)


def encode_image(image_path: str) -> np.ndarray:
    """
    Encode an image file into a 512-dim normalized CLIP embedding.

    Parameters
    ----------
    image_path : str — absolute or project-relative path to a JPEG/PNG image

    Returns
    -------
    np.ndarray shape (1, 512), dtype float32, L2-normalized
    """
    if not image_path:
        raise ValueError("image_path must be provided.")

    # Accept both absolute paths and notebook-relative paths (../data/images/...)
    path = Path(image_path)
    if not path.is_absolute():
        path = (NOTEBOOK_DIR / path).resolve()

    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    try:
        img = Image.open(path).convert("RGB")
    except Exception as e:
        raise IOError(f"Cannot open image {path}: {e}")

    inputs = clip_processor(images=[img], return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)

    with torch.no_grad():
        vis_out   = clip_model.vision_model(pixel_values=pixel_values)
        embedding = clip_model.visual_projection(vis_out.pooler_output)  # (1, 512)

    vec = embedding.cpu().float().numpy()
    return _l2_normalize(vec)


# ── Quick encoder tests ───────────────────────────────────────────────────────
t_vec = encode_text("test query")
assert t_vec.shape == (1, FAISS_DIM), f"unexpected text vec shape: {t_vec.shape}"
assert abs(np.linalg.norm(t_vec) - 1.0) < 1e-5, "text vec not normalized"
print(f"encode_text  → shape={t_vec.shape}, norm={np.linalg.norm(t_vec):.6f}  ✓")

# Pick a valid image from the dataset programmatically
sample_image_path = products_df["image_path"].iloc[0]
i_vec = encode_image(sample_image_path)
assert i_vec.shape == (1, FAISS_DIM), f"unexpected image vec shape: {i_vec.shape}"
assert abs(np.linalg.norm(i_vec) - 1.0) < 1e-5, "image vec not normalized"
print(f"encode_image → shape={i_vec.shape}, norm={np.linalg.norm(i_vec):.6f}  ✓")

encode_text  → shape=(1, 512), norm=1.000000  ✓


encode_image → shape=(1, 512), norm=1.000000  ✓


## 6. FAISS Search Helper

Low-level helper: query vector → list of `{faiss_index, pid, score}` dicts.

In [6]:
def _faiss_search(index: faiss.Index,
                  faiss_to_pid: dict,
                  query_vec: np.ndarray,
                  top_k: int) -> list[dict]:
    """
    Run a FAISS inner-product search and map results to PIDs.

    Returns
    -------
    list of dicts: [{faiss_index, pid, score}, ...], sorted by score desc.
    Entries with faiss_index == -1 or unmapped PIDs are silently skipped.
    """
    scores, faiss_indices = index.search(query_vec.astype(np.float32), top_k)
    scores        = scores[0]         # (top_k,)
    faiss_indices = faiss_indices[0]  # (top_k,)

    results = []
    for fidx, score in zip(faiss_indices, scores):
        if fidx == -1:
            continue
        pid = faiss_to_pid.get(int(fidx))
        if pid is None:
            continue
        if pid not in products_by_pid.index:
            continue
        results.append({"faiss_index": int(fidx),
                        "pid": pid,
                        "score": float(score)})

    return results  # already sorted descending by FAISS

## 7. Metadata Lookup and Result Formatting

In [7]:
def _attach_metadata(raw_results: list[dict],
                     score_col: str = "similarity_score") -> pd.DataFrame:
    """
    Attach product metadata to a list of raw FAISS results.

    Parameters
    ----------
    raw_results : list of {pid, score, ...}
    score_col   : name for the score column in the output DataFrame

    Returns
    -------
    pd.DataFrame with rank, pid, product_name, main_category,
                           brand, image_path, score_col
    """
    rows = []
    for rank, item in enumerate(raw_results, start=1):
        pid = item["pid"]
        meta = products_by_pid.loc[pid]
        rows.append({
            "rank"          : rank,
            "pid"           : pid,
            "product_name"  : meta["product_name"],
            "main_category" : meta["main_category"],
            "brand"         : meta.get("brand", "Unknown"),
            "image_path"    : meta["image_path"],
            score_col       : round(item["score"], 4),
        })
    return pd.DataFrame(rows)

## 8. Text Retrieval: `search_by_text`

In [8]:
def search_by_text(query: str, top_k: int = 10) -> pd.DataFrame:
    """
    Retrieve top-K products from the text FAISS index.

    Pipeline: query string → CLIP text encoder → normalize
              → text FAISS → map to PIDs → product metadata

    Returns
    -------
    pd.DataFrame sorted by similarity_score descending.
    """
    query_vec   = encode_text(query)
    raw_results = _faiss_search(text_index, text_faiss_to_pid, query_vec, top_k)
    return _attach_metadata(raw_results, score_col="text_score")


# ── Smoke test ────────────────────────────────────────────────────────────────
text_results = search_by_text("women's black leggings", top_k=5)
assert len(text_results) == 5
assert text_results["text_score"].is_monotonic_decreasing or \
       text_results["text_score"].iloc[0] >= text_results["text_score"].iloc[-1]
print("search_by_text — 'women's black leggings' (top 5):")
print(text_results[["rank","product_name","main_category","text_score"]].to_string(index=False))

search_by_text — 'women's black leggings' (top 5):
 rank                 product_name main_category  text_score
    1          NE Women's Leggings      Clothing      0.8098
    2          NE Women's Leggings      Clothing      0.7991
    3 Glam & Luxe Women's Leggings      Clothing      0.7980
    4 JRS Fashion Women's Leggings      Clothing      0.7682
    5 La Rochelle Women's Leggings      Clothing      0.7651


## 9. Image Retrieval: `search_by_image`

In [9]:
def search_by_image(image_path: str, top_k: int = 10) -> pd.DataFrame:
    """
    Retrieve top-K products from the image FAISS index.

    Pipeline: image file → CLIP image encoder → normalize
              → image FAISS → map to PIDs → product metadata

    Returns
    -------
    pd.DataFrame sorted by image_score descending.
    """
    query_vec   = encode_image(image_path)
    raw_results = _faiss_search(image_index, image_faiss_to_pid, query_vec, top_k)
    return _attach_metadata(raw_results, score_col="image_score")


# ── Smoke test — image selected programmatically from dataset ─────────────────
# Use the first valid image from the dataset (not hardcoded)
demo_image_path = products_df["image_path"].iloc[0]
demo_pid        = products_df["pid"].iloc[0]
demo_product    = products_df["product_name"].iloc[0]

print(f"Query image : {demo_image_path}")
print(f"Query PID   : {demo_pid}")
print(f"Product     : {demo_product}\n")

image_results = search_by_image(demo_image_path, top_k=5)
assert len(image_results) == 5
print("search_by_image — top 5 visually similar:")
print(image_results[["rank","product_name","main_category","image_score"]].to_string(index=False))
print(f"\nQuery product at rank 1: {image_results.iloc[0]['pid'] == demo_pid}  (expected True — same product)")

Query image : ../data/images/LJGDYQ2HQQGVHHXG.jpg
Query PID   : LJGDYQ2HQQGVHHXG
Product     : Sukuma Women's Leggings

search_by_image — top 5 visually similar:
 rank                      product_name main_category  image_score
    1           Sukuma Women's Leggings      Clothing       1.0000
    2           Kjaggs Women's Leggings      Clothing       0.9184
    3 medha Women's Multicolor Leggings      Clothing       0.8940
    4           Kjaggs Women's Leggings      Clothing       0.8938
    5            Famaya Girl's Leggings      Clothing       0.8929

Query product at rank 1: True  (expected True — same product)


## 10. Candidate Pool Builder

Merges text and image candidates by PID, preserving both modality scores.
No score fusion — that is the next stage.

In [10]:
def build_candidate_pool(text_candidates: pd.DataFrame | None,
                         image_candidates: pd.DataFrame | None) -> pd.DataFrame:
    """
    Merge text and image candidate sets into a unified candidate pool.

    Rules:
    - PID appears in both  → retrieved_by='both', both scores kept
    - PID in text only     → retrieved_by='text', image_score=NaN
    - PID in image only    → retrieved_by='image', text_score=NaN
    - No duplicate PIDs in output
    - Product metadata attached from products_ml_ready.csv

    Returns
    -------
    pd.DataFrame with columns:
        pid, product_name, main_category, brand, image_path,
        text_score, image_score, retrieved_by
    """
    if text_candidates is None and image_candidates is None:
        raise ValueError("At least one of text_candidates or image_candidates must be provided.")

    # Normalise column names so both DataFrames share 'score'
    def to_score_df(df, modality):
        score_col = "text_score" if modality == "text" else "image_score"
        cols = ["pid", score_col]
        return df[cols].rename(columns={score_col: f"{modality}_score"})

    if text_candidates is not None and image_candidates is not None:
        t = text_candidates[["pid", "text_score"]].copy()
        i = image_candidates[["pid", "image_score"]].copy()
        merged = t.merge(i, on="pid", how="outer")

        def _tag(row):
            has_text  = pd.notna(row["text_score"])
            has_image = pd.notna(row["image_score"])
            if has_text and has_image:
                return "both"
            elif has_text:
                return "text"
            return "image"

        merged["retrieved_by"] = merged.apply(_tag, axis=1)

    elif text_candidates is not None:
        merged = text_candidates[["pid", "text_score"]].copy()
        merged["image_score"]  = np.nan
        merged["retrieved_by"] = "text"

    else:
        merged = image_candidates[["pid", "image_score"]].copy()
        merged["text_score"]   = np.nan
        merged["retrieved_by"] = "image"

    # Attach product metadata
    meta_cols = ["pid", "product_name", "main_category", "brand", "image_path"]
    pool = merged.merge(products_df[meta_cols], on="pid", how="left")

    # Final column order
    pool = pool[["pid", "product_name", "main_category", "brand",
                 "image_path", "text_score", "image_score", "retrieved_by"]]

    assert pool["pid"].duplicated().sum() == 0, "duplicate PIDs in candidate pool"
    return pool.reset_index(drop=True)

## 11. Multimodal Retrieve — Unified Entry Point

Single function that dispatches to text-only, image-only, or combined retrieval.

In [11]:
def multimodal_retrieve(text_query: str | None = None,
                        image_path: str | None = None,
                        top_k: int = 20) -> pd.DataFrame:
    """
    Unified multimodal retrieval function.

    Cases
    -----
    Text only   : text_query provided, image_path=None
    Image only  : image_path provided, text_query=None
    Text+Image  : both provided — search both indexes, merge pool
    Neither     : raises ValueError

    Parameters
    ----------
    text_query : natural-language query string or None
    image_path : path to a product image file or None
    top_k      : number of candidates to retrieve per modality

    Returns
    -------
    pd.DataFrame — candidate pool with columns:
        pid, product_name, main_category, brand, image_path,
        text_score, image_score, retrieved_by
    """
    has_text  = text_query is not None and str(text_query).strip() != ""
    has_image = image_path is not None and str(image_path).strip() != ""

    if not has_text and not has_image:
        raise ValueError("Provide at least one of: text_query or image_path.")

    text_candidates  = None
    image_candidates = None

    if has_text:
        text_candidates = search_by_text(text_query, top_k=top_k)

    if has_image:
        image_candidates = search_by_image(image_path, top_k=top_k)

    return build_candidate_pool(text_candidates, image_candidates)

## 12. Execute — Case 1: Text Only

In [12]:
pd.set_option("display.max_colwidth", 45)

text_query = "men's formal shirt"
pool_text = multimodal_retrieve(text_query=text_query, top_k=10)

print(f"Case 1 — Text only: '{text_query}'")
print(f"Candidates returned : {len(pool_text)}")
print(f"retrieved_by values : {pool_text['retrieved_by'].unique().tolist()}")
print()
print(pool_text[["pid","product_name","main_category","text_score","retrieved_by"]].to_string(index=False))

Case 1 — Text only: 'men's formal shirt'
Candidates returned : 10
retrieved_by values : ['text']

             pid                                                             product_name main_category  text_score retrieved_by
SHTECFAKNASP6VVJ Jorzzer Roniya Men's Solid Formal, Party, Wedding, Casual, Festive Shirt      Clothing      0.6748         text
SHTEGYHSH7MG4SVU                                       Stylenara Men's Solid Casual Shirt      Clothing      0.6500         text
SHTEGZB8HYN9BCZK                                   Hoffmen Men's Self Design Formal Shirt      Clothing      0.6365         text
SHTEBG37THM95UTQ                                            Leaf Men's Solid Formal Shirt      Clothing      0.6343         text
WATE6S3AZNHMPTJA                                                  Sonata 77036SM02J Watch       Watches      0.6272         text
TOPEGQJYGPFHEKCK                     Reinvent Casual Short Sleeve Self Design Women's Top      Clothing      0.6191         text

## 13. Execute — Case 2: Image Only

In [13]:
# Select a valid image programmatically — pick a product from the Footwear category
footwear_row = products_df[products_df["main_category"].str.lower().str.contains("footwear", na=False)].iloc[0]
query_image_path = footwear_row["image_path"]
query_image_pid  = footwear_row["pid"]

pool_image = multimodal_retrieve(image_path=query_image_path, top_k=10)

print(f"Case 2 — Image only")
print(f"Query image path : {query_image_path}")
print(f"Query product    : {footwear_row['product_name']} ({footwear_row['main_category']})")
print(f"Candidates returned : {len(pool_image)}")
print(f"retrieved_by values : {pool_image['retrieved_by'].unique().tolist()}")
print()
print(pool_image[["pid","product_name","main_category","image_score","retrieved_by"]].to_string(index=False))

Case 2 — Image only
Query image path : ../data/images/SNDEDAPKZGGEGYHV.jpg
Query product    : S.m.a.R.T FEET Women Wedges (Footwear)
Candidates returned : 10
retrieved_by values : ['image']

             pid                                          product_name main_category  image_score retrieved_by
SNDEDAPKZGGEGYHV                           S.m.a.R.T FEET Women Wedges      Footwear       1.0000        image
SNDEJQ7Y56BXHTNM                                femitaly Women Bellies      Footwear       0.8945        image
SNDDY9RZPHET6BHX Lord's Antique Gold Women's Peeptoe Heels Women Heels      Footwear       0.8840        image
SHOEDP7ZGYFKGWNH                                        Bonzer Bellies      Footwear       0.8827        image
SHOECVMMYCMGFZZP                                     Wellworth Loafers      Footwear       0.8805        image
SHOEFECFE5BHMYZA                                         Imlee Mojaris      Footwear       0.8791        image
SNDEJPZMCWYT5QRW                

## 14. Execute — Case 3: Text + Image (Multimodal)

In [14]:
mm_text  = "women's leggings"

# Pick a clothing image programmatically
clothing_row     = products_df[products_df["main_category"].str.lower().str.contains("clothing", na=False)].iloc[0]
mm_image_path    = clothing_row["image_path"]

pool_mm = multimodal_retrieve(text_query=mm_text, image_path=mm_image_path, top_k=15)

both_count  = (pool_mm["retrieved_by"] == "both").sum()
text_only   = (pool_mm["retrieved_by"] == "text").sum()
image_only  = (pool_mm["retrieved_by"] == "image").sum()

print(f"Case 3 — Text + Image")
print(f"Text query   : '{mm_text}'")
print(f"Image path   : {mm_image_path}")
print(f"Total candidates : {len(pool_mm)}")
print(f"  retrieved by both  : {both_count}")
print(f"  text only          : {text_only}")
print(f"  image only         : {image_only}")
print()
print(pool_mm[["pid","product_name","main_category","text_score","image_score","retrieved_by"]].to_string(index=False))

Case 3 — Text + Image
Text query   : 'women's leggings'
Image path   : ../data/images/LJGDYQ2HQQGVHHXG.jpg
Total candidates : 25
  retrieved by both  : 5
  text only          : 10
  image only         : 10

             pid                           product_name main_category  text_score  image_score retrieved_by
ACBEHFPVV7PYCVHR             FIFO Bottom Women's  Combo      Clothing         NaN       0.8688        image
LJGDWAC3ZATUGA3D           La Rochelle Women's Leggings      Clothing      0.7264          NaN         text
LJGDYQ2HQQGVHHXG                Sukuma Women's Leggings      Clothing         NaN       1.0000        image
LJGDZJ625TFAJ7UA           Glam & Luxe Women's Leggings      Clothing      0.7784          NaN         text
LJGE2FAEGYYQAVWR                  Fexy Women's Leggings      Clothing      0.7068          NaN         text
LJGE3FPMVGJYUG8V                Ignite Women's Leggings      Clothing      0.7326          NaN         text
LJGE4YPSH5NK3GMY                Kjagg

## 15. Execute — Case 4: Error Handling (Neither Provided)

In [15]:
try:
    multimodal_retrieve()
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"Case 4 — No input correctly raises ValueError: {e}")

Case 4 — No input correctly raises ValueError: Provide at least one of: text_query or image_path.


## 16. Final Verification

In [16]:
def verify_pool(pool: pd.DataFrame, label: str):
    n_dup    = pool["pid"].duplicated().sum()
    bad_pids = [p for p in pool["pid"] if p not in products_by_pid.index]
    has_text_score  = "text_score"  in pool.columns
    has_image_score = "image_score" in pool.columns

    assert n_dup    == 0,  f"{label}: duplicate PIDs found"
    assert len(bad_pids) == 0, f"{label}: invalid PIDs: {bad_pids}"
    assert has_text_score and has_image_score, f"{label}: missing score columns"

    valid_scores = pool["text_score"].dropna().tolist() + pool["image_score"].dropna().tolist()
    assert all(isinstance(s, float) for s in valid_scores), f"{label}: non-numeric scores"
    print(f"  {label}: {len(pool)} candidates, 0 dup PIDs, 0 invalid PIDs, scores numeric  ✓")

print("=== Pool Verification ===")
verify_pool(pool_text,  "Text-only pool")
verify_pool(pool_image, "Image-only pool")
verify_pool(pool_mm,    "Multimodal pool")
print("All verifications passed.")

=== Pool Verification ===
  Text-only pool: 10 candidates, 0 dup PIDs, 0 invalid PIDs, scores numeric  ✓
  Image-only pool: 10 candidates, 0 dup PIDs, 0 invalid PIDs, scores numeric  ✓
  Multimodal pool: 25 candidates, 0 dup PIDs, 0 invalid PIDs, scores numeric  ✓
All verifications passed.


## 17. Final Report

In [17]:
print("=" * 55)
print("MULTIMODAL RETRIEVAL PIPELINE — FINAL REPORT")
print("=" * 55)
print(f"Products available      : {N_PRODUCTS}")
print(f"Text FAISS dimension    : {text_index.d}")
print(f"Image FAISS dimension   : {image_index.d}")
print(f"Text index size         : {text_index.ntotal}")
print(f"Image index size        : {image_index.ntotal}")
print(f"Text mapping rows       : {len(text_mapping_df)}")
print(f"Image mapping rows      : {len(image_mapping_df)}")
print()
print(f"Text retrieval (case 1) : {len(pool_text)} candidates")
print(f"Image retrieval (case 2): {len(pool_image)} candidates")
print(f"Multimodal pool (case 3): {len(pool_mm)} candidates")
print(f"  - retrieved by both   : {(pool_mm['retrieved_by']=='both').sum()}")
print(f"  - text only           : {(pool_mm['retrieved_by']=='text').sum()}")
print(f"  - image only          : {(pool_mm['retrieved_by']=='image').sum()}")
print()
print(f"Duplicate PIDs in pools : 0")
print(f"Invalid PIDs            : 0")
print(f"Normalized embeddings   : Yes (L2)")
print(f"CLIP model              : openai/clip-vit-base-patch32")
print(f"Similarity metric       : Inner Product (= cosine on normalized vecs)")
print()
print("Functions implemented:")
print("  encode_text(query)          → (1, 512) normalized")
print("  encode_image(path)          → (1, 512) normalized")
print("  search_by_text(query, k)    → DataFrame with text_score")
print("  search_by_image(path, k)    → DataFrame with image_score")
print("  build_candidate_pool(t, i)  → merged DataFrame, no dup PIDs")
print("  multimodal_retrieve(...)    → unified entry point")
print()
print("Status: COMPLETE")
print("Next stage: Score fusion / Re-ranking")
print("=" * 55)

MULTIMODAL RETRIEVAL PIPELINE — FINAL REPORT
Products available      : 4681
Text FAISS dimension    : 512
Image FAISS dimension   : 512
Text index size         : 4681
Image index size        : 4681
Text mapping rows       : 4681
Image mapping rows      : 4681

Text retrieval (case 1) : 10 candidates
Image retrieval (case 2): 10 candidates
Multimodal pool (case 3): 25 candidates
  - retrieved by both   : 5
  - text only           : 10
  - image only          : 10

Duplicate PIDs in pools : 0
Invalid PIDs            : 0
Normalized embeddings   : Yes (L2)
CLIP model              : openai/clip-vit-base-patch32
Similarity metric       : Inner Product (= cosine on normalized vecs)

Functions implemented:
  encode_text(query)          → (1, 512) normalized
  encode_image(path)          → (1, 512) normalized
  search_by_text(query, k)    → DataFrame with text_score
  search_by_image(path, k)    → DataFrame with image_score
  build_candidate_pool(t, i)  → merged DataFrame, no dup PIDs
  multimo